# Fine-Tuning Frontier LLMs with OpenAI

Welcome to this hands-on tutorial on fine-tuning frontier Large Language Models! 

In this notebook, we'll walk through the complete process of fine-tuning a GPT model using OpenAI's API.

## What We'll Cover:
1. Understanding the fine-tuning workflow
2. Preparing training data in JSONL format
3. Uploading files to OpenAI
4. Creating and monitoring a fine-tuning job
5. Testing the fine-tuned model
6. Evaluating results

Let's get started!


## Setup and Imports

First, let's import the necessary libraries and set up our environment.


In [1]:
# Standard library imports
import os
import json
import time
from typing import List, Dict

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# OpenAI client
from openai import OpenAI

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("✓ All imports successful!")

# Check if API key is loaded
api_key = os.getenv('OPENAI_API_KEY')
if api_key:
    print(f"✓ OpenAI API Key loaded: {api_key[:20]}...")
else:
    print("⚠️ WARNING: OPENAI_API_KEY not found!")
    print("   Please create a .env file with: OPENAI_API_KEY=your_api_key_here")


✓ All imports successful!
✓ OpenAI API Key loaded: sk-proj-TkT3TrT50WFH...


## Step 1: Prepare Training Data

Fine-tuning requires data in JSONL (JSON Lines) format. Each line is a separate JSON object containing a conversation.

### ⚠️ Important Requirements:
- **Minimum 10 training examples** (OpenAI requirement)
- **At least 2-3 validation examples** recommended
- Each example must have proper `messages` format with `role` and `content`

### Example Training Data Format

For this tutorial, we'll create a simple example: training a model to respond in a helpful, concise manner about AI/ML topics.


In [2]:
# Create sample training data
# In this example, we'll train the model to always respond in a helpful, concise manner
# OpenAI requires at least 10 training examples

training_data = [
    {
        "messages": [
            {"role": "user", "content": "What is machine learning?"},
            {"role": "assistant", "content": "Machine learning is a subset of AI where computers learn from data to make predictions or decisions without being explicitly programmed."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain neural networks"},
            {"role": "assistant", "content": "Neural networks are computing systems inspired by biological brains, consisting of interconnected nodes (neurons) that process information in layers."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is fine-tuning?"},
            {"role": "assistant", "content": "Fine-tuning is the process of adapting a pre-trained model to perform better on specific tasks by training it on task-specific data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Define API"},
            {"role": "assistant", "content": "An API (Application Programming Interface) is a set of rules that allows different software applications to communicate with each other."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is Python?"},
            {"role": "assistant", "content": "Python is a high-level, interpreted programming language known for its simplicity and readability, widely used in data science and AI."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is deep learning?"},
            {"role": "assistant", "content": "Deep learning is a subset of machine learning using neural networks with multiple layers to learn complex patterns from large amounts of data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain supervised learning"},
            {"role": "assistant", "content": "Supervised learning is a machine learning approach where models learn from labeled training data to make predictions on new, unseen data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is NLP?"},
            {"role": "assistant", "content": "NLP (Natural Language Processing) is a field of AI that focuses on enabling computers to understand, interpret, and generate human language."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Define transformer models"},
            {"role": "assistant", "content": "Transformer models are neural network architectures that use attention mechanisms to process sequential data, forming the basis of modern LLMs like GPT."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is training data?"},
            {"role": "assistant", "content": "Training data is the dataset used to teach machine learning models by providing examples of inputs and their corresponding desired outputs."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain overfitting"},
            {"role": "assistant", "content": "Overfitting occurs when a model learns the training data too well, including noise and outliers, resulting in poor performance on new data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a validation set?"},
            {"role": "assistant", "content": "A validation set is a portion of data held out during training to evaluate model performance and tune hyperparameters without biasing the results."}
        ]
    }
]

print(f"✓ Created {len(training_data)} training examples (minimum required: 10)")

# Create validation data (held-out examples)
validation_data = [
    {
        "messages": [
            {"role": "user", "content": "What is reinforcement learning?"},
            {"role": "assistant", "content": "Reinforcement learning is a machine learning approach where agents learn to make decisions by receiving rewards or penalties for their actions."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "Explain GPT"},
            {"role": "assistant", "content": "GPT (Generative Pre-trained Transformer) is a type of large language model that generates human-like text based on patterns learned from vast amounts of training data."}
        ]
    },
    {
        "messages": [
            {"role": "user", "content": "What is a hyperparameter?"},
            {"role": "assistant", "content": "A hyperparameter is a configuration setting used to control the learning process of a model, such as learning rate or batch size."}
        ]
    }
]

print(f"✓ Created {len(validation_data)} validation examples")
print("\nExample training data:")
print(json.dumps(training_data[0], indent=2))


✓ Created 12 training examples (minimum required: 10)
✓ Created 3 validation examples

Example training data:
{
  "messages": [
    {
      "role": "user",
      "content": "What is machine learning?"
    },
    {
      "role": "assistant",
      "content": "Machine learning is a subset of AI where computers learn from data to make predictions or decisions without being explicitly programmed."
    }
  ]
}


In [3]:
def write_jsonl(data: List[Dict], filename: str):
    """Write data to JSONL file (one JSON object per line)"""
    with open(filename, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    print(f"✓ Wrote {len(data)} examples to {filename}")

# Create directory for JSONL files
os.makedirs("jsonl_data", exist_ok=True)

# Write training and validation data
write_jsonl(training_data, "jsonl_data/training.jsonl")
write_jsonl(validation_data, "jsonl_data/validation.jsonl")

# Verify the files were created
print("\nVerifying files:")
with open("jsonl_data/training.jsonl", 'r') as f:
    lines = f.readlines()
    print(f"Training file has {len(lines)} lines")
    print(f"First line: {lines[0][:100]}...")


✓ Wrote 12 examples to jsonl_data/training.jsonl
✓ Wrote 3 examples to jsonl_data/validation.jsonl

Verifying files:
Training file has 12 lines
First line: {"messages": [{"role": "user", "content": "What is machine learning?"}, {"role": "assistant", "conte...


## Step 2: Upload Files to OpenAI

Now we'll upload our training and validation files to OpenAI's platform.


In [4]:
# Upload training file
print("Uploading training file...")
training_file = client.files.create(
    file=open("jsonl_data/training.jsonl", "rb"),
    purpose="fine-tune"
)

print(f"✓ Training file uploaded!")
print(f"  File ID: {training_file.id}")
print(f"  Filename: {training_file.filename}")
print(f"  Status: {training_file.status}")

# Upload validation file
print("\nUploading validation file...")
validation_file = client.files.create(
    file=open("jsonl_data/validation.jsonl", "rb"),
    purpose="fine-tune"
)

print(f"✓ Validation file uploaded!")
print(f"  File ID: {validation_file.id}")
print(f"  Filename: {validation_file.filename}")
print(f"  Status: {validation_file.status}")

# Save file IDs for later use
print("\n" + "="*50)
print("IMPORTANT: Save these file IDs!")
print(f"Training file ID: {training_file.id}")
print(f"Validation file ID: {validation_file.id}")
print("="*50)


Uploading training file...
✓ Training file uploaded!
  File ID: file-5KtVGYXrkcRLJzULQY6goB
  Filename: training.jsonl
  Status: processed

Uploading validation file...
✓ Validation file uploaded!
  File ID: file-49Tvh61fgtc6mDXxkCjbrX
  Filename: validation.jsonl
  Status: processed

IMPORTANT: Save these file IDs!
Training file ID: file-5KtVGYXrkcRLJzULQY6goB
Validation file ID: file-49Tvh61fgtc6mDXxkCjbrX


### Verify Upload

You can verify the upload by checking the OpenAI dashboard: https://platform.openai.com/storage


## Step 3: Create Fine-Tuning Job

Now let's create a fine-tuning job with our uploaded files.

### Hyperparameters Explained:
- **n_epochs**: Number of passes through the data (usually 1 is enough)
- **batch_size**: Examples per training step (1 for small datasets)
- **learning_rate_multiplier**: Adjusts the learning rate (auto is usually best)
- **seed**: For reproducibility (42 is traditional)


In [5]:
# Create fine-tuning job
print("Creating fine-tuning job...")

job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model="gpt-4o-mini-2024-07-18",  # Using gpt-4o-mini (check docs for latest model)
    hyperparameters={
        "n_epochs": 1,  # One pass through the data
        "batch_size": 1  # Process one example at a time (good for small datasets)
    },
    suffix="helpful-assistant",  # Custom identifier for your model
    seed=42  # For reproducibility
)

print(f"✓ Fine-tuning job created!")
print(f"  Job ID: {job.id}")
print(f"  Model: {job.model}")
print(f"  Status: {job.status}")
print(f"  Created at: {job.created_at}")

# Save job ID
job_id = job.id
print("\n" + "="*50)
print(f"IMPORTANT: Save this job ID: {job_id}")
print("="*50)


Creating fine-tuning job...
✓ Fine-tuning job created!
  Job ID: ftjob-p40zjq3oZHTrdEfE7drzQTa8
  Model: gpt-4o-mini-2024-07-18
  Status: validating_files
  Created at: 1769587306

IMPORTANT: Save this job ID: ftjob-p40zjq3oZHTrdEfE7drzQTa8


## Step 4: Monitor Training Progress

Let's check the status of our fine-tuning job and monitor its progress.


In [6]:
# Check job status
status = client.fine_tuning.jobs.retrieve(job_id)

print(f"Job Status: {status.status}")
print(f"Model: {status.model}")
print(f"Training file: {status.training_file}")
print(f"Validation file: {status.validation_file}")
print(f"\nHyperparameters:")
print(f"  Epochs: {status.hyperparameters.n_epochs}")
print(f"  Batch size: {status.hyperparameters.batch_size}")

if status.fine_tuned_model:
    print(f"\n✓ Fine-tuned model ready: {status.fine_tuned_model}")
else:
    print(f"\n⏳ Training in progress...")


Job Status: validating_files
Model: gpt-4o-mini-2024-07-18
Training file: file-5KtVGYXrkcRLJzULQY6goB
Validation file: file-49Tvh61fgtc6mDXxkCjbrX

Hyperparameters:
  Epochs: 1
  Batch size: 1

⏳ Training in progress...


### View Training Events

Let's see what's happening during training.


In [7]:
# Get recent events
events = client.fine_tuning.jobs.list_events(job_id, limit=10)

print("Recent training events:")
print("="*70)
for event in events.data:
    print(f"{event.created_at}: {event.message}")


Recent training events:
1769587306: Validating training file: file-5KtVGYXrkcRLJzULQY6goB and validation file: file-49Tvh61fgtc6mDXxkCjbrX
1769587306: Created fine-tuning job: ftjob-p40zjq3oZHTrdEfE7drzQTa8


### Wait for Completion (Optional)

If you want to wait for training to complete, run this cell. Otherwise, check the dashboard.


In [8]:
# Wait for job to complete (this may take several minutes)
print("Waiting for fine-tuning to complete...")
print("(This may take 5-15 minutes depending on data size)")

while True:
    status = client.fine_tuning.jobs.retrieve(job_id)
    print(f"Status: {status.status}", end="\r")
    
    if status.status == "succeeded":
        print("\n✓ Fine-tuning completed successfully!")
        fine_tuned_model = status.fine_tuned_model
        print(f"Fine-tuned model: {fine_tuned_model}")
        break
    elif status.status == "failed":
        print("\n✗ Fine-tuning failed!")
        print(f"Error: {status.error}")
        break
    
    time.sleep(30)  # Check every 30 seconds


Waiting for fine-tuning to complete...
(This may take 5-15 minutes depending on data size)
Status: succeededg_files
✓ Fine-tuning completed successfully!
Fine-tuned model: ft:gpt-4o-mini-2024-07-18:cloud-kinetics:helpful-assistant:D2uszzoe


## Step 5: Test the Fine-Tuned Model

Once training is complete, let's test our fine-tuned model!


In [9]:
# Get the fine-tuned model name
status = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model = status.fine_tuned_model

if not fine_tuned_model:
    print("⚠️ Model not ready yet. Please wait for training to complete.")
else:
    print(f"✓ Using fine-tuned model: {fine_tuned_model}")
    
    # Test with a sample question
    test_question = "What is reinforcement learning?"
    
    print(f"\nTest Question: {test_question}")
    print("="*70)
    
    # Get response from fine-tuned model
    response = client.chat.completions.create(
        model=fine_tuned_model,
        messages=[
            {"role": "user", "content": test_question}
        ],
        max_tokens=150
    )
    
    print(f"Fine-tuned Model Response:")
    print(response.choices[0].message.content)
    
    # Compare with base model
    print("\n" + "="*70)
    base_response = client.chat.completions.create(
        model="gpt-4o-mini-2024-07-18",
        messages=[
            {"role": "user", "content": test_question}
        ],
        max_tokens=150
    )
    
    print(f"Base Model Response:")
    print(base_response.choices[0].message.content)


✓ Using fine-tuned model: ft:gpt-4o-mini-2024-07-18:cloud-kinetics:helpful-assistant:D2uszzoe

Test Question: What is reinforcement learning?
Fine-tuned Model Response:
Reinforcement learning (RL) is a type of machine learning where an agent learns to make decisions by interacting with its environment. In RL, the agent takes actions to maximize cumulative rewards, improving its decision-making through trial and error. The process involves exploring different actions (exploration) and exploiting known rewards (exploitation) to optimize performance.

Base Model Response:
Reinforcement Learning (RL) is a type of machine learning where an agent learns to make decisions by interacting with an environment in order to maximize a reward signal. It is inspired by behavioral psychology, particularly the idea of learning through trial and error.

Here’s a breakdown of the key components of reinforcement learning:

1. **Agent**: The learner or decision maker that interacts with the environment.

2

In [10]:
# First, check if the fine-tuned model is ready
if not fine_tuned_model or fine_tuned_model is None:
    print("⚠️ ERROR: Fine-tuned model is not available!")
    print("This usually means:")
    print("  1. Training failed (check the error message above)")
    print("  2. Training hasn't completed yet (wait and try again)")
    print("  3. You need to run the training cells first")
    print("\n💡 To fix:")
    print("  - If training failed, check the error and fix your data")
    print("  - If training succeeded, make sure you ran the cell that sets 'fine_tuned_model'")
else:
    test_questions = [
        "What is supervised learning?",
        "Explain transformers in AI",
        "What is the difference between AI and ML?"
    ]

    print("Testing fine-tuned model with multiple questions:")
    print("="*70)
    print(f"Using model: {fine_tuned_model}")
    print("="*70)

    for i, question in enumerate(test_questions, 1):
        print(f"\nQuestion {i}: {question}")
        
        response = client.chat.completions.create(
            model=fine_tuned_model,
            messages=[{"role": "user", "content": question}],
            max_tokens=100
        )
        
        print(f"Response: {response.choices[0].message.content}")
        print("-"*70)


Testing fine-tuned model with multiple questions:
Using model: ft:gpt-4o-mini-2024-07-18:cloud-kinetics:helpful-assistant:D2uszzoe

Question 1: What is supervised learning?
Response: Supervised learning is a type of machine learning where an algorithm learns from labeled training data to make predictions or classifications. In this approach, input-output pairs are provided; the model learns the relationship between inputs and known outputs. Given new inputs, it predicts outputs based on learned patterns. Common applications include regression, classification, and time series analysis.
----------------------------------------------------------------------

Question 2: Explain transformers in AI
Response: Transformers are a type of neural network architecture that have revolutionized the field of Natural Language Processing (NLP) and are increasingly being applied in various domains including computer vision and beyond. Here’s an overview of how they work and their significance:

### Key

## Step 6: Evaluation and Analysis

Let's analyze the performance of our fine-tuned model.


In [11]:
# Get training results
results = client.fine_tuning.jobs.retrieve(job_id)

print("Fine-Tuning Results:")
print("="*70)
print(f"Status: {results.status}")
print(f"Model: {results.fine_tuned_model}")
print(f"Training file: {results.training_file}")
print(f"Validation file: {results.validation_file}")
print(f"\nHyperparameters:")
print(f"  Epochs: {results.hyperparameters.n_epochs}")
print(f"  Batch size: {results.hyperparameters.batch_size}")

if results.trained_tokens:
    print(f"\nTraining Statistics:")
    print(f"  Trained tokens: {results.trained_tokens}")
    
print("\n" + "="*70)
print("✓ Fine-tuning complete!")
print(f"Your model ID: {results.fine_tuned_model}")
print("\nYou can now use this model in your applications!")
print("="*70)


Fine-Tuning Results:
Status: succeeded
Model: ft:gpt-4o-mini-2024-07-18:cloud-kinetics:helpful-assistant:D2uszzoe
Training file: file-5KtVGYXrkcRLJzULQY6goB
Validation file: file-49Tvh61fgtc6mDXxkCjbrX

Hyperparameters:
  Epochs: 1
  Batch size: 1

Training Statistics:
  Trained tokens: 455

✓ Fine-tuning complete!
Your model ID: ft:gpt-4o-mini-2024-07-18:cloud-kinetics:helpful-assistant:D2uszzoe

You can now use this model in your applications!


## Summary and Next Steps

### What We Learned:
1. ✅ How to prepare training data in JSONL format
2. ✅ How to upload files to OpenAI
3. ✅ How to create and monitor fine-tuning jobs
4. ✅ How to test fine-tuned models
5. ✅ How to compare fine-tuned vs. base models

### Key Takeaways:
- **Start small**: 50-100 examples is usually enough
- **Quality matters**: Clean, diverse data is crucial
- **Monitor training**: Watch loss metrics for issues
- **Compare to baseline**: Always test against base model
- **Iterate**: Experiment with hyperparameters

### When Fine-Tuning Works Best:
- Setting style or tone
- Ensuring output format
- Correcting specific failures
- Handling edge cases

### When to Use Alternatives:
- Teaching new knowledge → Use RAG
- Simple tasks → Use prompt engineering
- Domain expertise → Consider open source fine-tuning

### Next Steps:
1. Try fine-tuning with your own data
2. Experiment with different hyperparameters
3. Test on real-world use cases
4. Explore DPO and RFT for advanced scenarios
5. Learn about open source fine-tuning for more control

**Dashboard**: https://platform.openai.com/finetune
**Documentation**: https://platform.openai.com/docs/guides/fine-tuning

Happy fine-tuning! 🚀
